# screamingface · Quickstart

Compose several AI models into one **fusion**, run it on a benchmark sample, and see whether the
panel beats its strongest member. Connect → pick → compose → run → compare, in five small cells.

This checked-in run is an explicit **SIMULATION**: it uses 20 synthetic, GPQA-shaped science
questions and deterministic local model adapters, so GitHub can render a safe and reproducible
example. It is not a provider benchmark and it does not use or reveal gated GPQA questions.

**Going live.** Replace the setup call below with `sf.setup()` and have three things ready:

1. **A running AI Gateway.** From this repository:
   `cd apps/aigateway && uv sync && uv run uvicorn aigateway.main:app --port 9105`.
   `sf.setup()` discovers `http://127.0.0.1:9105` automatically; for any other host, pass
   `gateway="..."` or set `SCREAMINGFACE_GATEWAY_URL`.
2. **Connected providers.** Gateway login unlocks your encrypted credential vault; the setup
   panel then connects providers with OAuth or an API key. `sf.models.list()` shows models from
   your actively connected providers only.
3. **Hugging Face access for real GPQA.** Install the dataset extra (`uv sync --extra datasets`),
   accept the gated terms at `huggingface.co/datasets/Idavidrein/gpqa`, and log in with
   `huggingface-cli login` (or set `HF_TOKEN`). This HF login is separate from gateway login.

Live mode never silently falls back to simulation.

When running from a repository checkout, launch this notebook with
`uv run --extra notebook jupyter lab examples/00_quickstart.ipynb`, or select the interpreter at
`packages/screamingface/.venv/bin/python` in your editor.

## 1 · Connect

In [1]:
import sys

import screamingface as sf

if not hasattr(sf, "setup"):
    raise RuntimeError(
        "Wrong notebook kernel: select packages/screamingface/.venv/bin/python "
        "or launch Jupyter with `uv run --extra notebook jupyter lab`. "
        f"Current Python: {sys.executable}"
    )

session = sf.setup(mode="mock", static_widgets=True)
session

SetupPanel(state='connected', credentials=<never stored>)

## 2 · Pick models

In [2]:
available = sf.models.list(max_price=20)
available

['codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6']

## 3 · Compose a URL4-backed fusion

In [3]:
fusion = sf.Fusion(
    "frontier-trio",
    models=available[:3],
    reduce="majority_vote",
    judge=available[0],
)
fusion

Role,Model
Judge,codex/gpt-5.5
Member,gemini-cli/gemini-2.5-pro
Member,anthropic/claude-sonnet-4-6


The normal display keeps the recipe readable. Ask for the canonical, shareable URL4 explicitly:

In [4]:
fusion.url4

"(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=frontier-trio;sf_judge=codex/gpt-5.5"

## 4 · Run

Each member answers each question exactly once through an embedded URL4 node — the `url4`
package's node facade running inside this process, with no extra server to start. The fusion vote
and best-member baseline reuse those same answers, so the comparison does not spend twice. In live
mode, evaluation first checks that every required provider is connected and every model is
available. A blocked run fails once with `FusionNotReady` before loading benchmark data or making
model calls.

In [5]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=frontier-trio;sf_judge=codex/gpt-5.5", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='frontier-trio', reduce='majority_vote', judge='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='estimate:SDK catalog', pricing_as_of='2026-07-16', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0)), failures=())

## 5 · Compare

In [6]:
{
    "mode": run.mode,
    "provenance": run.dataset_source,
    "sample_size": run.sample_size,
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
    "cost_usd": run.cost_usd,
}

{'mode': 'mock',
 'provenance': 'synthetic-gpqa-shaped',
 'sample_size': 20,
 'score': 100.0,
 'baseline': 80.0,
 'gain': 20.0,
 'cost_usd': 0.0}

> **Interpretation:** gain is the fusion score minus the strongest member score,
using the same panel answers. The number above demonstrates the SDK flow only; because this
checked-in execution is simulated, it is not evidence that these named providers achieve these
scores on GPQA.

**Next:** keep a lineup in a reviewable file with the YAML companion,
[`yaml_quickstart.ipynb`](yaml_quickstart.ipynb) — or share this exact fusion by sending its
`fusion.url4` recipe.